SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved. <br>
SPDX-License-Identifier: Apache-2.0 <br>

Licensed under the Apache License, Version 2.0 (the "License"); <br>
you may not use this file except in compliance with the License. <br>
You may obtain a copy of the License at <br>

http://www.apache.org/licenses/LICENSE-2.0  

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, <br>
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. <br>
See the License for the specific language governing permissions and <br>
limitations under the License. 

In [1]:
import time

import torch
from torch import nn

import warp as wp

from warpkan.torch_ext.kanlinear import KANLinear
from efficient_kan import KAN as PKAN
wp.init()


Warp 1.3.1 initialized:
   CUDA Toolkit 12.5, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX A6000" (48 GiB, sm_86, mempool enabled)
     "cuda:1"   : "NVIDIA RTX A6000" (48 GiB, sm_86, mempool enabled)
   CUDA peer access:
     Supported fully (all-directional)
   Kernel cache:
     /home/amoradzadeh/.cache/warp/1.3.1


# MLP vs KAN

In [2]:
in_dim = out_dim = 5
mlp_hidden = 24
grid_num = 46
kan = nn.Sequential(KANLinear(in_dim, 32, 3, grid_num, [-5, 5], has_mlp=False), KANLinear(32, in_dim, 3, grid_num, [-5, 5], has_mlp=False)).cuda()
public_kan = PKAN([in_dim, 32, in_dim], grid_size=grid_num, grid_range=[-5, 5]).cuda()

mlp = nn.Sequential(nn.Linear(in_dim, mlp_hidden), nn.ReLU(), nn.Linear(mlp_hidden, mlp_hidden), nn.ReLU(), nn.Linear(mlp_hidden, in_dim)).cuda()
print("KAN params: ", sum(p.numel() for p in kan.parameters()))
print("MLP params: ", sum(p.numel() for p in mlp.parameters()))

KAN params:  16640
MLP params:  869


In [3]:
x = torch.randn(1024, in_dim).cuda()
x_val = torch.randn(1024, in_dim).cuda()
freq = torch.linspace(1, 7, in_dim).cuda()
target = torch.sin( x * freq)

target_val = torch.sin( x_val * freq)

In [4]:
for model in [kan, public_kan, mlp]:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    print(model)
    for epoch in range(5000):
        optimizer.zero_grad()
        y = model(x)
        loss = torch.mean((target-y)**2)
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            if epoch == 0:
                print(f"{'epoch':5} | {'train_loss':12} | {'val_loss':12}")    
            print(f"{epoch:5} | {loss.item():0.10f} | {torch.mean((target_val- model(x_val))**2).item():0.10f}")

Sequential(
  (0): KANLinear(
    (bspline): BSpline()
  )
  (1): KANLinear(
    (bspline): BSpline()
  )
)
Module bspline_fwd_3_5_32_55 85fa624 load on device 'cuda:0' took 0.26 ms  (cached)
Module bspline_fwd_3_32_5_55 e22be0f load on device 'cuda:0' took 0.18 ms  (cached)
Module bspline_bwd_3_32_5_55 b196a86 load on device 'cuda:0' took 0.37 ms  (cached)
Module bspline_bwd_3_5_32_55 fe79caa load on device 'cuda:0' took 0.22 ms  (cached)
epoch | train_loss   | val_loss    
    0 | 0.4894213378 | 0.4930798709
  500 | 0.0006244293 | 0.0020518959
 1000 | 0.0001161052 | 0.0009989508
 1500 | 0.0000469034 | 0.0007729410
 2000 | 0.0000250232 | 0.0006737955
 2500 | 0.0000150238 | 0.0006191116
 3000 | 0.0000095776 | 0.0005849323
 3500 | 0.0000063248 | 0.0005618229
 4000 | 0.0000042745 | 0.0005451597
 4500 | 0.0000029324 | 0.0005319791
KAN(
  (layers): ModuleList(
    (0-1): 2 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
epoch | train_loss   | val_loss    
    0 | 0.4910612106 | 0.

## warp KAN vs efficient public KAN

* Cost is proportional to order not grid_dim O(grid_dim * rest) -> O(order * rest)
* Memory efficient 


In [3]:
x = torch.randn(2**20, in_dim).cuda()


In [4]:
%timeit kan(x)

Module bspline_fwd_3_5_32_55 85fa624 load on device 'cuda:0' took 0.42 ms  (cached)
Module bspline_fwd_3_32_5_55 e22be0f load on device 'cuda:0' took 7.04 ms  (cached)
12 ms ± 124 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
from efficient_kan import KAN as PKAN
public_kan = PKAN([in_dim, 10, in_dim], grid_size=grid_num, grid_range=[-5, 5]).cuda()
for _ in range(100):
    public_kan(x)

torch.cuda.synchronize()


In [13]:
%timeit public_kan(x)

239 ms ± 6.89 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Param Size vs KAN:

* cost does not depend on warp KAN parameters directly 

In [8]:
for grid_num in [32, 64, 128, 256, 512, 1024]:
    kan = KAN([in_dim, 10, in_dim], grid_num, 3, grid_range=[-5, 5]).cuda()
    for _ in range(100):
        kan(x)
    torch.cuda.synchronize()
    s = time.perf_counter()
    for _ in range(100):
        kan(x)
    torch.cuda.synchronize()
    e = time.perf_counter()
    print("KAN params: ", sum(p.numel() for p in kan.parameters()), ", dt: ", 10*(e - s))
    

NameError: name 'KAN' is not defined

# old

In [14]:
for grid_num in [32, 64, 128, 256, 512, 1024]:
    kan = KAN([in_dim, 10, in_dim], grid_num, 3, grid_range=[-5, 5]).cuda()
    for _ in range(100):
        kan(x)
    torch.cuda.synchronize()
    s = time.perf_counter()
    for _ in range(100):
        kan(x)
    torch.cuda.synchronize()
    e = time.perf_counter()
    print("KAN params: ", sum(p.numel() for p in kan.parameters()), ", dt: ", 10*(e - s))
    

Module bspline_fwd_3_5_10_41 0949564 load on device 'cuda:0' took 556.17 ms
Module bspline_fwd_3_10_5_41 32a553d load on device 'cuda:0' took 781.57 ms
KAN params:  3900 , dt:  7.505270359106362
Module bspline_fwd_3_5_10_73 75799a6 load on device 'cuda:0' took 497.35 ms
Module bspline_fwd_3_10_5_73 8a5b2c1 load on device 'cuda:0' took 767.40 ms
KAN params:  7100 , dt:  7.6986676175147295
Module bspline_fwd_3_5_10_137 50c69a1 load on device 'cuda:0' took 492.56 ms
Module bspline_fwd_3_10_5_137 606f531 load on device 'cuda:0' took 791.22 ms
KAN params:  13500 , dt:  7.738256407901645
Module bspline_fwd_3_5_10_265 2956176 load on device 'cuda:0' took 483.87 ms
Module bspline_fwd_3_10_5_265 1add2f7 load on device 'cuda:0' took 747.81 ms
KAN params:  26300 , dt:  7.723037791438401
Module bspline_fwd_3_5_10_521 66415b4 load on device 'cuda:0' took 466.32 ms
Module bspline_fwd_3_10_5_521 a2f4b54 load on device 'cuda:0' took 744.80 ms
KAN params:  51900 , dt:  7.80763397924602
Module bspline_f

In [15]:
in_dim = out_dim = 5
for grid_num in [32, 64, 128, 256, 512, 1024]:
    public_kan = PKAN([in_dim, 10, in_dim], grid_size=grid_num, grid_range=[-5, 5]).cuda()
    try:
        for _ in range(100):
            public_kan(x)
        torch.cuda.synchronize()
        s = time.perf_counter()
        for _ in range(100):
            public_kan(x)
        torch.cuda.synchronize()
        e = time.perf_counter()
        print("Public KAN params: ", sum(p.numel() for p in kan.parameters()), ", dt: ", 10*(e - s))
    except:
        print("Public KAN params: ", sum(p.numel() for p in kan.parameters()), " failed!")
        break
        
    

Public KAN params:  103100 , dt:  177.2430953430012
Public KAN params:  103100 , dt:  319.93099936749786
Public KAN params:  103100  failed!
